In [ ]:
#| default_exp state

# State and diffs

> Track execution freshness, downstream staleness, and noise-free cell diffs.

Reading a notebook is risky when saved outputs come from stale code. This module tracks which cells match their last recorded run and renders clean diffs across code and prose.

In [ ]:
#| export
import ast, builtins, difflib, json, subprocess, symtable
from dataclasses import dataclass, field
from pathlib import Path

from fastcore.nbio import dict2nb, read_nb

from nbskill.core import cell_source, chapters, clean_python_source, project_root, source_hash

In [ ]:
from tempfile import TemporaryDirectory
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.test import test_eq

## Execution freshness and dependencies

Each non-empty code cell receives a `CellState` comparing its current `source_hash` against `metadata.nbskill.executed_hash`. Static name reads connect downstream cells to the latest earlier cell defining each name, while dynamic namespace access or in-place mutation stays visible under `uncertainty`.

In [ ]:
#| export
_DYNAMIC_CALLS = frozenset({"eval", "exec", "globals", "locals", "vars", "setattr", "delattr"})
_BUILTIN_NAMES = frozenset(dir(builtins))

@dataclass
class CellState:
    "Execution status and static name dependencies for one code cell."
    cell_id: str
    cell_idx: int
    state: str  # 'clean', 'dirty', 'stale', 'error', or 'unknown'
    defines: list[str] = field(default_factory=list)
    reads: list[str] = field(default_factory=list)
    depends_on: list[str] = field(default_factory=list)
    stale_from: list[str] = field(default_factory=list)
    uncertainty: list[str] = field(default_factory=list)
    source_hash: str = ""
    executed_hash: str = ""

@dataclass
class NotebookState:
    "Execution freshness summary for one notebook."
    path: str
    cells: list[CellState]
    counts: dict[str, int]
    limits: list[str]

    def by_id(self) -> dict[str, CellState]: return {c.cell_id: c for c in self.cells}

    def __str__(self) -> str:
        parts = ", ".join(f"{k}={v}" for k, v in self.counts.items() if v)
        lines = [f"{self.path}: {parts or 'no code cells'}"]
        for c in self.cells:
            if c.state in ("dirty", "stale", "error"):
                via = f" (from {', '.join(c.stale_from)})" if c.stale_from else ""
                lines.append(f"- {c.cell_id}: {c.state}{via}")
        return "\n".join(lines)

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

`cell_dependencies` inspects one cell with `symtable` and `ast` to separate defined names, referenced globals, and dynamic uncertainty flags.

In [ ]:
#| export
def _nested_global_reads(table) -> set[str]:
    reads = set()
    for child in table.get_children():
        reads.update(s.get_name() for s in child.get_symbols() if s.is_global() and s.is_referenced())
        reads.update(_nested_global_reads(child))
    return reads

def _dynamic_reasons(tree: ast.AST) -> list[str]:
    reasons = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in _DYNAMIC_CALLS:
            reasons.add("dynamic-namespace")
        targets = node.targets if isinstance(node, (ast.Assign, ast.Delete)) else [node.target] if isinstance(node, (ast.AnnAssign, ast.AugAssign)) else []
        if any(isinstance(part, (ast.Attribute, ast.Subscript)) for t in targets for part in ast.walk(t)): reasons.add("in-place-mutation")
    return sorted(reasons)

def cell_dependencies(cell) -> tuple[list[str], list[str], list[str]]:
    "Return `(defines, reads, uncertainty)` for one code cell."
    cleaned = clean_python_source(cell_source(cell))
    if not cleaned or not cleaned.strip(): return [], [], []
    try:
        tree = ast.parse(cleaned)
        table = symtable.symtable(cleaned, f"<cell {cell.get('id', '')}>", "exec")
    except SyntaxError: return [], [], ["syntax-error"]
    syms = table.get_symbols()
    defines = sorted({s.get_name() for s in syms if s.is_assigned() or s.is_imported() or s.is_namespace()})
    reads = sorted(({s.get_name() for s in syms if s.is_referenced()} | _nested_global_reads(table)) - _BUILTIN_NAMES)
    return defines, reads, _dynamic_reasons(tree)

`notebook_state` wires those cell facts together, marks cells whose stored `executed_hash` lags behind their current source as `dirty`, and propagates `stale` status to downstream readers.

In [ ]:
#| export
def _direct_state(cell) -> tuple[str, str, str]:
    cur = source_hash(cell_source(cell), length=None)
    meta = (cell.get("metadata", {}) or {}).get("nbskill") or {}
    executed = str(meta.get("executed_hash", "")) if isinstance(meta, dict) else ""
    has_err = any(o.get("output_type") == "error" for o in (cell.get("outputs") or []))
    if executed == cur: return ("error" if has_err else "clean"), cur, executed
    return ("dirty" if executed else "unknown"), cur, executed

def notebook_state(path: str | Path) -> NotebookState:
    "Return execution freshness and likely downstream staleness for `path` without running code."
    p = Path(path).expanduser()
    nb = read_nb(p)
    records, providers = [], {}
    for idx, cell in enumerate(nb.cells):
        if cell.get("cell_type") != "code" or not cell_source(cell).strip(): continue
        defines, reads, unc = cell_dependencies(cell)
        state, cur, exe = _direct_state(cell)
        deps = sorted({providers[name] for name in reads if name in providers})
        cid = str(cell.get("id", f"cell-{idx}"))
        records.append(CellState(cid, idx, state, defines, reads, deps, [], unc, cur[:12], exe[:12]))
        for name in defines: providers[name] = cid
    by_id = {r.cell_id: r for r in records}
    changed = True
    while changed:
        changed = False
        for r in records:
            if r.state in ("dirty", "error"): continue
            roots = {s for d in r.depends_on for s in ([d] if by_id[d].state in ("dirty", "error") else by_id[d].stale_from)}
            if roots and (r.state != "stale" or sorted(roots) != r.stale_from):
                r.state, r.stale_from, changed = "stale", sorted(roots), True
    counts = {s: sum(r.state == s for r in records) for s in ("clean", "dirty", "stale", "error", "unknown")}
    limits = ["Static name dependencies are conservative.", "Dynamic namespace access and in-place mutation can hide impact."]
    return NotebookState(str(p), records, counts, limits)

Editing an executed upstream cell marks it `dirty` and flags every dependent downstream cell as `stale` while preserving dynamic uncertainty.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "flow.ipynb"
    cells = [mk_cell("raw = 1"), mk_cell("clean = raw + 1"), mk_cell("globals()['k'] = clean")]
    for c in cells[:2]: c.metadata = {"nbskill": {"executed_hash": source_hash(cell_source(c), length=None)}}
    write_nb(new_nb(cells), p)
    test_eq([c.state for c in notebook_state(p).cells], ["clean", "clean", "unknown"])
    cells[0].source = "raw = 2"
    write_nb(new_nb(cells), p)
    after = notebook_state(p)
    test_eq(([c.state for c in after.cells], after.cells[2].uncertainty), (["dirty", "stale", "stale"], ["dynamic-namespace", "in-place-mutation"]))

## Noise-free notebook diffs

Raw `.ipynb` diffs drown source changes in JSON escaping, execution counts, and stored outputs. `nb_diff_lines` renders each cell with its stable ID and chapter name so `diff_nb` compares only meaningful content.

In [ ]:
#| export
def nb_diff_lines(nb, *, include_markdown: bool = True) -> list[str]:
    "Render notebook cells into plain lines with stable cell IDs and chapter labels for diffing."
    lines = []
    for chap in chapters(nb):
        for cell in chap.cells:
            ctype = cell.get("cell_type", "")
            if ctype == "raw" or (ctype == "markdown" and not include_markdown): continue
            cid = cell.get("id", "")
            lines.append(f"### cell {cid} ({ctype}) [{chap.title}]")
            lines.extend(cell_source(cell).splitlines())
            lines.append("")
    return lines

`diff_nb` compares a notebook against a Git revision (defaulting to `HEAD`) or renders an untracked notebook as additions. Markdown prose is included by default so reviewers see changed explanations alongside code.

In [ ]:
#| export
def _git_nb(path: Path, ref: str):
    root = project_root(path)
    rel = path.resolve().relative_to(root).as_posix()
    res = subprocess.run(["git", "-C", str(root), "show", f"{ref}:{rel}"], capture_output=True, text=True)
    return dict2nb(json.loads(res.stdout)) if res.returncode == 0 else None

def diff_nb(
    path: str | Path,  # Notebook to diff
    ref_a: str | None = "HEAD",  # Base Git revision, or `None` for empty baseline
    ref_b: str | None = None,  # Target Git revision, or `None` for working tree
    *,
    include_markdown: bool = True,  # Include Markdown explanations alongside code cells
) -> str:
    "Return a cell-level unified diff without JSON, output, or metadata noise."
    p = Path(path).expanduser()
    nb_a = _git_nb(p, ref_a) if ref_a else None
    nb_b = _git_nb(p, ref_b) if ref_b else (read_nb(p) if p.is_file() else None)
    lines_a = nb_diff_lines(nb_a, include_markdown=include_markdown) if nb_a else []
    lines_b = nb_diff_lines(nb_b, include_markdown=include_markdown) if nb_b else []
    label_a, label_b = f"{ref_a or 'empty'}:{p.name}", f"{ref_b or 'worktree'}:{p.name}"
    diff = list(difflib.unified_diff(lines_a, lines_b, fromfile=label_a, tofile=label_b, lineterm=""))
    return "\n".join(diff)

Comparing a disposable notebook against an empty baseline shows how `include_markdown` toggles prose visibility while excluding stored outputs.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "note.ipynb"
    c = mk_cell("answer = 42", outputs=[dict(output_type="stream", name="stdout", text="noisy\n")])
    write_nb(new_nb([mk_cell("## Intro\nExplain the contract.", cell_type="markdown"), c]), p)
    full_diff = diff_nb(p, ref_a=None, include_markdown=True)
    code_diff = diff_nb(p, ref_a=None, include_markdown=False)
    assert "Explain the contract." in full_diff and "Explain the contract." not in code_diff
    assert "+answer = 42" in code_diff and "noisy" not in full_diff
full_diff